In [ ]:
from bng_simulator.utils.logger_utils import load_log_data
import numpy as np

# # Make matplotlib plots interactive (allows zoom/pan, etc.)
# %matplotlib widget
# import matplotlib.pyplot as plt

In [ ]:
# Run number
run = 9
metadata, data = load_log_data(run)

In [ ]:
metadata.keys()

In [ ]:
data.keys()

In [ ]:
log_data = data[('ego', 'gtstate')]
log_data = {k: np.array(v) for k, v in log_data.items()}

In [ ]:
for k, v in log_data.items():
    print(k, v.shape)

In [ ]:
import matplotlib
matplotlib.use("Qt5Agg")  # Or another interactive backend like "TkAgg", Qt5Agg
# matplotlib.use("widget")  # Or another interactive backend like "TkAgg", Qt5Agg
import matplotlib.pyplot as plt

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math

# Use time as the shared x-axis.
time = log_data["time"]

# Compute average of the rear wheelspeed angVelB (average of wheelRL_angVelB and wheelRR_angVelB).
if "wheelRL_angVelB" in log_data and "wheelRR_angVelB" in log_data:
    rear_angVelB_avg = (np.array(log_data["wheelRL_angVelB"]) + np.array(log_data["wheelRR_angVelB"])) / 2.0
else:
    rear_angVelB_avg = None

# Engine RPM.
engine_RPM = log_data["RPM"] if "RPM" in log_data else None

# Boost Pressure from turboBoost key.
boost_pressure = log_data["turboBoost"] if "turboBoost" in log_data else None

throttleValve = log_data["throttleValve"] if "throttleValve" in log_data else None

# Gear Ratio.
gear_ratio = log_data["gearRatio"] if "gearRatio" in log_data else None

# Engine Torque.
engineTorque = np.array(log_data["engineTorque"]) if "engineTorque" in log_data else None

# Average Rear Propulsion Torque (average of wheelRL_propTorque and wheelRR_propTorque).
if "wheelRL_propTorque" in log_data and "wheelRR_propTorque" in log_data:
    rear_propTorque_avg = (np.array(log_data["wheelRL_propTorque"]) + np.array(log_data["wheelRR_propTorque"])) / 2.0
else:
    rear_propTorque_avg = None

# Throttle.
throttle = log_data["throttle"] if "throttle" in log_data else None

# Collection of quantities to plot.
quantities = {
    "Avg Rear Wheelspeed angVelB": rear_angVelB_avg,
    "Engine RPM": engine_RPM,
    "Boost Pressure": boost_pressure,
    "Gear Ratio": gear_ratio,
    "Engine Torque": engineTorque,
    "Avg Rear Prop Torque": rear_propTorque_avg,
    "Throttle": throttle,
    "throttleValve": throttleValve
}

n_plots = len(quantities)
n_cols = 2
n_rows = math.ceil(n_plots / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, sharex=True, figsize=(14, n_rows * 3))
axes = axes.flatten()

for ax, (label, values) in zip(axes, quantities.items()):
    if values is not None:
        ax.plot(time, values)
    else:
        ax.text(0.5, 0.5, f"{label} not available", ha="center", transform=ax.transAxes)
    ax.set_title(label)
    ax.grid(True)

# Remove any unused subplots.
for ax in axes[n_plots:]:
    fig.delaxes(ax)

axes[-1].set_xlabel("Time")
plt.tight_layout()
# plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math
from scipy.signal import savgol_filter

# Use time as the shared x-axis.
time = log_data["time"]
dt = np.mean(np.diff(time))

# Compute average of the rear wheelspeed angVelB (average of wheelRL_angVelB and wheelRR_angVelB).
if "wheelRL_angVelB" in log_data and "wheelRR_angVelB" in log_data:
    rear_angVelB_avg = (np.array(log_data["wheelRL_angVelB"]) + np.array(log_data["wheelRR_angVelB"])) / 2.0
else:
    rear_angVelB_avg = None
rear_angVelB_avg_der = np.gradient(rear_angVelB_avg, dt)

# Engine RPM.
engine_RPM = log_data["RPM"] if "RPM" in log_data else None
# Let's convert in rad/s
engine_RPM = engine_RPM * 2 * np.pi / 60
engine_rpm_der = np.gradient(engine_RPM, dt)

# Turbo Boost Pressure.
boost_pressure = log_data["turboBoost"] if "turboBoost" in log_data else None

# Gear Ratio.
gear_ratio = log_data["gearRatio"] if "gearRatio" in log_data else None

# Engine Torque.
engineTorque = np.array(log_data["engineTorque"]) if "engineTorque" in log_data else None

# Average Rear Propulsion Torque (average of wheelRL_propTorque and wheelRR_propTorque).
if "wheelRL_propTorque" in log_data and "wheelRR_propTorque" in log_data:
    rear_propTorque_avg = (np.array(log_data["wheelRL_propTorque"]) + np.array(log_data["wheelRR_propTorque"])) / 2.0
else:
    rear_propTorque_avg = None

# Throttle.
throttle = log_data["throttle"] if "throttle" in log_data else None

# Collection of quantities to plot.
quantities = {
    "Avg Rear Wheelspeed angVelB": rear_angVelB_avg,
    "Engine RPM": engine_RPM,
    # "Boost Pressure": boost_pressure,
    "Gear Ratio": gear_ratio,
    "Engine Torque": engineTorque,
    # "Avg Rear Prop Torque": rear_propTorque_avg,
    # "Throttle": throttle,
    "Engine RPM Derivative": engine_rpm_der,
    "Avg Rear Der": rear_angVelB_avg_der
}

### Different Savitzky–Golay parameters per quantity ###
# Window length must be odd and <= number of samples in each array.
# You can tune these based on trial and error.
filter_params = {
    "Avg Rear Wheelspeed angVelB": {"window_length": 21, "polyorder": 4},
    "Engine RPM": {"window_length": 21, "polyorder": 4},
    "Boost Pressure": {"window_length": 21,  "polyorder": 4},
    "Engine RPM Derivative": {"window_length": 21, "polyorder": 4},
    "Avg Rear Der": {"window_length": 101, "polyorder": 4},
    # "Gear Ratio": {"window_length": 51,  "polyorder": 2},
    # "Engine Torque": {"window_length": 101, "polyorder": 3},
    # "Avg Rear Prop Torque": {"window_length": 101, "polyorder": 3},
    # "Throttle": {"window_length": 59, "polyorder": 2},
}

# Apply filtering.
filtered_quantities = {}
for label, data_array in quantities.items():
    if data_array is not None:
        params = filter_params.get(label, {})
        if len(params) == 0:
            print(f"No filter parameters found for {label}. Skipping filtering.")
            filtered_quantities[label] = data_array
            continue
        # Ensure window length is smaller than length of data and is odd.
        effective_window = min(params["window_length"], len(data_array))
        if effective_window % 2 == 0:
            effective_window -= 1
        if effective_window < 3:
            effective_window = 3
        filtered_quantities[label] = savgol_filter(
            data_array,
            window_length=effective_window,
            polyorder=params["polyorder"],
            delta=dt,
        )
    else:
        filtered_quantities[label] = None

n_plots = len(quantities)
n_cols = 2
n_rows = math.ceil(n_plots / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, sharex=True, figsize=(14, n_rows * 3))
axes = axes.flatten()

for i, (label, values) in enumerate(quantities.items()):
    ax = axes[i]
    if values is not None:
        ax.plot(time, values, label="Original", alpha=0.5)
        ax.plot(time, filtered_quantities[label], label="Filtered", linewidth=2)
    else:
        ax.text(0.5, 0.5, f"{label} not available", ha="center", transform=ax.transAxes)
    ax.set_title(label)
    ax.grid(True)
    ax.legend(loc="upper right")

# Remove any unused subplots.
for ax in axes[n_plots:]:
    fig.delaxes(ax)

axes[-1].set_xlabel("Time")
plt.tight_layout()
plt.show()
# At gear ratio one, gets 96 rad/s wr, and 817.5 rad/s run 9, at t=102.2
# gear ratio 2.22, gets 29.8 rad/s wr, and 563.5 rad/s  
# Engine torque = 168, RMP der = 14
# inv = 1 / 8.516 eta, J_eng = 1

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math
from scipy.signal import butter, filtfilt

# A simple low-pass filter using Butterworth + filtfilt
def lowpass_filter(data, cutoff_hz, fs, order=4):
    # Nyquist frequency:
    nyquist = 0.5 * fs
    cutoff_norm = cutoff_hz / nyquist
    b, a = butter(order, cutoff_norm, btype='low', analog=False)
    return filtfilt(b, a, data)

# Suppose we already have time, dt, and log_data from earlier steps
# dt = np.mean(np.diff(time))
fs = 1.0 / dt  # sample frequency in Hz

# Example cutoff frequencies per quantity (adjust as needed)
lowpass_params = {
    "Avg Rear Wheelspeed angVelB": 20.0,  # cutoff Hz
    "Engine RPM": 50.0,
    "Boost Pressure": 30.0,
    # "Gear Ratio": 1.0,
    # "Engine Torque": 5.0,
    # "Avg Rear Prop Torque": 5.0,
    # "Throttle": 2.0
}

# Using the same 'quantities' dict from before (assumed already defined)
filtered_quantities = {}
for label, values in quantities.items():
    if values is not None:
        cutoff = lowpass_params.get(label, None)
        if cutoff is None:
            print(f"No cutoff frequency found for {label}. Skipping filtering.")
            filtered_quantities[label] = values
            continue
        # Filter the data
        filtered_quantities[label] = lowpass_filter(values, cutoff_hz=cutoff, fs=fs, order=4)
    else:
        filtered_quantities[label] = None

# Plot original vs filtered side by side
n_plots = len(quantities)
n_cols = 2
n_rows = math.ceil(n_plots / n_cols)
fig, axes = plt.subplots(n_rows, n_cols, sharex=True, figsize=(14, n_rows * 3))
axes = axes.flatten()

for i, (label, values) in enumerate(quantities.items()):
    ax = axes[i]
    if values is not None:
        ax.plot(time, values, label="Original", alpha=0.5)
        ax.plot(time, filtered_quantities[label], label="Filtered (Low-pass)", linewidth=2)
    else:
        ax.text(0.5, 0.5, f"{label} not available", ha="center", transform=ax.transAxes)
    ax.set_title(label)
    ax.grid(True)
    ax.legend(loc="upper right")

# Remove any unused subplots.
for ax in axes[n_plots:]:
    fig.delaxes(ax)

axes[-1].set_xlabel("Time")
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math

def exponential_filter(data, alpha=0.1):
    """
    data: array of data to filter
    alpha: filter coefficient (0 < alpha <= 1)
           smaller alpha -> more smoothing
    """
    filtered = np.zeros_like(data)
    filtered[0] = data[0]
    for i in range(1, len(data)):
        filtered[i] = alpha * data[i] + (1 - alpha) * filtered[i - 1]
    return filtered

time = log_data["time"]

# Use time, dt, and quantities as previously defined
# Example alpha values per quantity
alpha_params = {
    "Avg Rear Wheelspeed angVelB": 0.2,
    "Engine RPM": 0.8,
    "Boost Pressure": 0.3,
    "Avg Rear Der": 0.05,
    "Engine RPM Derivative": 0.2,
    # "Gear Ratio": 0.1,
    # "Engine Torque": 0.15,
    # "Avg Rear Prop Torque": 0.15,
    # "Throttle": 0.2
}

filtered_quantities = {}
for label, values in quantities.items():
    if values is not None:
        alpha = alpha_params.get(label, None)
        if alpha is None:
            print(f"No alpha specified for {label}. Skipping filtering.")
            filtered_quantities[label] = values
        else:
            filtered_quantities[label] = exponential_filter(values, alpha=alpha)
    else:
        filtered_quantities[label] = None

# Plot original vs filtered side by side
n_plots = len(quantities)
n_cols = 2
n_rows = math.ceil(n_plots / n_cols)
fig, axes = plt.subplots(n_rows, n_cols, sharex=True, figsize=(14, n_rows * 3))
axes = axes.flatten()

for i, (label, values) in enumerate(quantities.items()):
    ax = axes[i]
    if values is not None:
        ax.plot(time, values, label="Original", alpha=0.5)
        ax.plot(time, filtered_quantities[label], label="Filtered (Exponential)", linewidth=2)
    else:
        ax.text(0.5, 0.5, f"{label} not available", ha="center", transform=ax.transAxes)
    ax.set_title(label)
    ax.grid(True)
    ax.legend(loc="upper right")

# Remove any unused subplots.
for ax in axes[n_plots:]:
    fig.delaxes(ax)

axes[-1].set_xlabel("Time")
plt.tight_layout()
plt.show()

In [ ]:
def plot_data(data, x_key="time", y_keys=None, title="", figsize=(12, 8)):
    """
    Plots several data keys as a function of another key (generally time).
    
    Args:
        data (dict): Dictionary containing data arrays.
        x_key (str): Key to use for the x-axis (default is "time").
        y_keys (list, optional): List of keys to plot on the y-axis. 
            If None, all keys except x_key will be used.
        title (str, optional): Title for the entire figure.
        figsize (tuple, optional): Size of the figure.
    
    Raises:
        ValueError: If no valid y_keys are found.
    """
    # Determine which keys to plot if not explicitly provided.
    if y_keys is None:
        y_keys = [k for k in data.keys() if k != x_key]
    
    n_plots = len(y_keys)
    if n_plots == 0:
        raise ValueError("No y_keys provided or found in data.")

    # Decide a suitable grid based on the number of plots.
    ncols = int(np.ceil(np.sqrt(n_plots)))
    nrows = int(np.ceil(n_plots / ncols))
    
    fig, axs = plt.subplots(nrows, ncols, figsize=figsize, squeeze=False,sharex=True)
    
    # Plot each y_key against the x_key.
    for idx, key in enumerate(y_keys):
        row = idx // ncols
        col = idx % ncols
        ax = axs[row][col]
        ax.plot(data[x_key], data[key])
        ax.set_xlabel(x_key)
        ax.set_ylabel(key)
        ax.set_title(key)
    
    # Hide any unused subplots.
    for idx in range(n_plots, nrows * ncols):
        row = idx // ncols
        col = idx % ncols
        axs[row][col].set_visible(False)
    
    if title:
        fig.suptitle(title)
    plt.tight_layout()
    plt.show()

In [ ]:
# Assuming log_data is a dictionary mapping field names to numpy arrays
time = log_data['time']
rpm = log_data['RPM']
engine_torque = log_data['engineTorque']
gearbox_torque = log_data['gearboxTorque']
throttle = log_data['throttle']
engine_load = log_data['engineLoad']

# Plot 1: Engine Torque vs RPM
plt.figure(figsize=(10, 6))
plt.scatter(rpm, engine_torque, s=10, alpha=0.5)
plt.title("Engine Torque vs RPM")
plt.xlabel("RPM")
plt.ylabel("Engine Torque")
plt.grid(True)

# Plot 2: Engine Torque vs Throttle (colored by Engine Load)
plt.figure(figsize=(10, 6))
scatter = plt.scatter(throttle, engine_torque, s=10, alpha=0.5, c=engine_load, cmap='viridis')
plt.title("Engine Torque vs Throttle")
plt.xlabel("Throttle")
plt.ylabel("Engine Torque")
plt.colorbar(scatter, label='Engine Load')
plt.grid(True)

# Plot 3: Engine and Gearbox Torque over Time
plt.figure(figsize=(10, 6))
plt.plot(time, engine_torque, label='Engine Torque', alpha=0.7)
plt.plot(time, gearbox_torque, label='Gearbox Torque', alpha=0.7)
plt.title("Torque over Time")
plt.xlabel("Time (s)")
plt.ylabel("Torque")
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
# Extract relevant fields
rpm = log_data['RPM']
engine_torque = log_data['engineTorque']
throttle = log_data['throttle']
brake = log_data['brake']
time = log_data['time']

# Remove any NaN values if needed
valid_idx = ~np.isnan(rpm) & ~np.isnan(engine_torque) & ~np.isnan(throttle)
rpm = rpm[valid_idx]
engine_torque = engine_torque[valid_idx]
throttle = throttle[valid_idx]

# Plot 1: Engine Torque vs RPM colored by throttle
plt.figure(figsize=(10, 6))
scatter = plt.scatter(rpm, engine_torque, c=throttle, cmap='viridis', s=20, alpha=0.7)
plt.title("Engine Torque vs RPM (colored by Throttle)")
plt.xlabel("RPM")
plt.ylabel("Engine Torque")
cbar = plt.colorbar(scatter)
cbar.set_label("Throttle")
plt.grid(True)

# Alternatively, if you want to color by engine torque itself:
# scatter = plt.scatter(rpm, engine_torque, c=engine_torque, cmap='magma', s=20, alpha=0.7)
# plt.title("Engine Torque vs RPM (colored by Engine Torque)")
# cbar.set_label("Engine Torque")

# Plot 2: Throttle and Brake vs Time
plt.figure(figsize=(10, 6))
plt.plot(time, throttle, label="Throttle", color='blue', linewidth=1.5)
plt.plot(time, brake, label="Brake", color='red', linewidth=1.5)
plt.title("Throttle and Brake vs Time")
plt.xlabel("Time (s)")
plt.ylabel("Input value")
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
time = log_data['time']
accel_x = log_data['accel_x']
accel_y = log_data['accel_y']
accel_z = log_data['accel_z']
angVel_z = log_data['angVel_z']

plt.figure(figsize=(10, 16))

plt.subplot(4, 1, 1)
plt.plot(time, accel_x, label="Accel X", color='b')
plt.xlabel("Time (s)")
plt.ylabel("Acceleration X (m/s²)")
plt.grid(True)

plt.subplot(4, 1, 2)
plt.plot(time, accel_y, label="Accel Y", color='g')
plt.xlabel("Time (s)")
plt.ylabel("Acceleration Y (m/s²)")
plt.grid(True)

plt.subplot(4, 1, 3)
plt.plot(time, accel_z, label="Accel Z", color='r')
plt.xlabel("Time (s)")
plt.ylabel("Acceleration Z (m/s²)")
plt.grid(True)

plt.subplot(4, 1, 4)
plt.plot(time, angVel_z, label="Angular Velocity Z", color='m')
plt.xlabel("Time (s)")
plt.ylabel("Angular Velocity Z (rad/s)")
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
from scipy.signal import savgol_filter

# Extract the raw data
time = log_data['time']
accel_x = log_data['accel_x']
accel_y = log_data['accel_y']
accel_z = log_data['accel_z']
angVel_z = log_data['angVel_z']

# Filter parameters: window length must be an odd number; polyorder < window_length
window_length = 101  # Adjust as needed for smoothing
polyorder = 3

# Apply Savitzky-Golay filter to smooth the data
filtered_accel_x = savgol_filter(accel_x, window_length, polyorder)
filtered_accel_y = savgol_filter(accel_y, window_length, polyorder)
filtered_accel_z = savgol_filter(accel_z, window_length, polyorder)
filtered_angVel_z = savgol_filter(angVel_z, window_length, polyorder)

plt.figure(figsize=(10, 16))

plt.subplot(4, 1, 1)
plt.plot(time, filtered_accel_x, label="Filtered Accel X", color='b')
plt.xlabel("Time (s)")
plt.ylabel("Acceleration X (m/s²)")
plt.grid(True)
plt.legend()

plt.subplot(4, 1, 2)
plt.plot(time, filtered_accel_y, label="Filtered Accel Y", color='g')
plt.xlabel("Time (s)")
plt.ylabel("Acceleration Y (m/s²)")
plt.grid(True)
plt.legend()

plt.subplot(4, 1, 3)
plt.plot(time, filtered_accel_z, label="Filtered Accel Z", color='r')
plt.xlabel("Time (s)")
plt.ylabel("Acceleration Z (m/s²)")
plt.grid(True)
plt.legend()

plt.subplot(4, 1, 4)
plt.plot(time, filtered_angVel_z, label="Filtered Angular Velocity Z", color='m')
plt.xlabel("Time (s)")
plt.ylabel("Angular Velocity Z (rad/s)")
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def exponential_moving_average(data, alpha):
    """
    Compute the exponential moving average of a 1D array.

    Args:
        data (np.ndarray): Input data array.
        alpha (float): Smoothing factor between 0 and 1. Higher alpha means less smoothing.
    
    Returns:
        np.ndarray: Filtered data array.
    """
    ema = np.zeros_like(data)
    ema[0] = data[0]
    for i in range(1, len(data)):
        ema[i] = alpha * data[i] + (1 - alpha) * ema[i - 1]
    return ema

# Smoothing parameter; tweak as necessary
alpha = 0.9

# Get raw signals from log_data
time = log_data['time']
accel_x = log_data['accel_x']
accel_y = log_data['accel_y']
accel_z = log_data['accel_z']
angVel_z = log_data['angVel_z']

# Apply exponential moving average filter to signals
filtered_accel_x = exponential_moving_average(accel_x, alpha)
filtered_accel_y = exponential_moving_average(accel_y, alpha)
filtered_accel_z = exponential_moving_average(accel_z, alpha)
filtered_angVel_z = exponential_moving_average(angVel_z, alpha)

plt.figure(figsize=(10, 16))

_ax1 = plt.subplot(4, 1, 1)
plt.plot(time, filtered_accel_x, label="Filtered Accel X", color='b')
plt.xlabel("Time (s)")
plt.ylabel("Acceleration X (m/s²)")
plt.grid(True)
plt.legend()

_ax2 = plt.subplot(4, 1, 2, sharex=_ax1)
plt.plot(time, filtered_accel_y, label="Filtered Accel Y", color='g')
plt.xlabel("Time (s)")
plt.ylabel("Acceleration Y (m/s²)")
plt.grid(True)
plt.legend()

plt.subplot(4, 1, 3, sharex=_ax1)
plt.plot(time, filtered_accel_z, label="Filtered Accel Z", color='r')
plt.xlabel("Time (s)")
plt.ylabel("Acceleration Z (m/s²)")
plt.grid(True)
plt.legend()

plt.subplot(4, 1, 4, sharex=_ax1)
plt.plot(time, throttle, label="Filtered Angular Velocity Z", color='m')
plt.xlabel("Time (s)")
plt.ylabel("Angular Velocity Z (rad/s)")
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
def exponential_moving_average(data, alpha):
    """
    Compute the exponential moving average of a 1D array.

    Args:
        data (np.ndarray): Input data array.
        alpha (float): Smoothing factor between 0 and 1. Higher alpha means less smoothing.
    
    Returns:
        np.ndarray: Filtered data array.
    """
    ema = np.zeros_like(data)
    ema[0] = data[0]
    for i in range(1, len(data)):
        ema[i] = alpha * data[i] + (1 - alpha) * ema[i - 1]
    return ema

# Smoothing parameters for comparison
alpha_values = [0.05, 0.1, 0.2]

# Get raw signals from log_data
time = log_data['time']
accel_x = log_data['accel_x']

plt.figure(figsize=(10, 8))
plt.plot(time, accel_x, label="Raw Accel X", color='gray', alpha=0.5)

for alpha in alpha_values:
    filtered = exponential_moving_average(accel_x, alpha)
    plt.plot(time, filtered, label=f"EMA (alpha={alpha})")

plt.title("Raw vs. EMA Filtered Acceleration X")
plt.xlabel("Time (s)")
plt.ylabel("Acceleration X (m/s²)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Select a smoothing parameter (tweak as needed)
# dt = 0.005  # Time step in seconds
# mem_tau = 0.03  # Memory time constant in seconds
# alpha = dt / (mem_tau + dt)
alpha = 0.25
print(f"Smoothing parameter (alpha): {alpha}")

# Get raw signals from log_data
time = log_data['time']
accel_x = log_data['accel_x']
accel_y = log_data['accel_y']
accel_z = log_data['accel_z']
angVel_z = log_data['angVel_z']

# Apply exponential moving average filter to each signal
filtered_accel_x = exponential_moving_average(accel_x, alpha)
filtered_accel_y = exponential_moving_average(accel_y, alpha)
filtered_accel_z = exponential_moving_average(accel_z, alpha)
filtered_angVel_z = exponential_moving_average(angVel_z, alpha)

plt.figure(figsize=(10, 16))

# Plot for Acceleration X
_ax1 = plt.subplot(4, 1, 1)
plt.plot(time, accel_x, label="Raw Accel X", color='lightblue', alpha=0.6)
plt.plot(time, filtered_accel_x, label="Filtered Accel X", color='b', linewidth=2)
plt.xlabel("Time (s)")
plt.ylabel("Accel X (m/s²)")
plt.grid(True)
plt.legend()

# Plot for Acceleration Y
_ax2 = plt.subplot(4, 1, 2, sharex=_ax1)
plt.plot(time, accel_y, label="Raw Accel Y", color='lightgreen', alpha=0.6)
plt.plot(time, filtered_accel_y, label="Filtered Accel Y", color='g', linewidth=2)
plt.xlabel("Time (s)")
plt.ylabel("Accel Y (m/s²)")
plt.grid(True)
plt.legend()

# Plot for Acceleration Z
_ax3 = plt.subplot(4, 1, 3, sharex=_ax1)
plt.plot(time, accel_z, label="Raw Accel Z", color='salmon', alpha=0.6)
plt.plot(time, filtered_accel_z, label="Filtered Accel Z", color='r', linewidth=2)
plt.xlabel("Time (s)")
plt.ylabel("Accel Z (m/s²)")
plt.grid(True)
plt.legend()

# Plot for Angular Velocity Z
_ax4 = plt.subplot(4, 1, 4, sharex=_ax1)
plt.plot(time, angVel_z, label="Raw AngVel Z", color='plum', alpha=0.6)
plt.plot(time, filtered_angVel_z, label="Filtered AngVel Z", color='m', linewidth=2)
plt.xlabel("Time (s)")
plt.ylabel("AngVel Z (rad/s)")
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
time = log_data['time']
rpm = log_data['RPM']
throttle = log_data['throttle']
engine_load = log_data['engineLoad']
gear_ratio = log_data['gearRatio']
engine_torque = log_data['engineTorque']
boost = log_data['turboBoost']

plt.figure(figsize=(12, 15))

ax1 = plt.subplot(6, 1, 1)
plt.plot(time, rpm, label="RPM", color='green')
plt.ylabel("RPM")
plt.grid(True)
plt.legend()

ax2 = plt.subplot(6, 1, 2, sharex=ax1)
plt.plot(time, throttle, label="Throttle", color='blue')
plt.ylabel("Throttle")
plt.grid(True)
plt.legend()

ax3 = plt.subplot(6, 1, 3, sharex=ax1)
plt.plot(time, engine_load, label="Engine Load", color='red')
plt.ylabel("Engine Load")
plt.grid(True)
plt.legend()

ax4 = plt.subplot(6, 1, 4, sharex=ax1)
plt.plot(time, gear_ratio, label="Gear Ratio", color='purple')
plt.ylabel("Gear Ratio")
plt.grid(True)
plt.legend()

ax5 = plt.subplot(6, 1, 5, sharex=ax1)
plt.plot(time, engine_torque, label="Engine Torque", color='orange')
plt.xlabel("Time (s)")
plt.ylabel("Engine Torque")
plt.grid(True)
plt.legend()

ax6 = plt.subplot(6, 1, 6, sharex=ax1)
plt.plot(time, boost, label="Turbo Boost", color='brown')
plt.xlabel("Time (s)")
plt.ylabel("Turbo Boost")
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Extract time from log_data
time = log_data['time']

# Extract position components. These keys should exist in your log_data.
pos_x = log_data['pos_x']
pos_y = log_data['pos_y']
pos_z = log_data['pos_z']

# Extract velocity components. Adjust keys if case differs.
vel_x = log_data['vel_x']
vel_y = log_data['vel_y']
vel_z = log_data['vel_z']

# Extract wheelspeed for each wheel.
wheelFR_speed = log_data['wheelFR_speed']
wheelFL_speed = log_data['wheelFL_speed']
wheelRR_speed = log_data['wheelRR_speed']
wheelRL_speed = log_data['wheelRL_speed']

plt.figure(figsize=(12, 15))

# Plot Position x-y-z
ax1 = plt.subplot(3, 1, 1)
plt.plot(time, pos_x, label="Pos X", color='red')
plt.plot(time, pos_y, label="Pos Y", color='green')
plt.plot(time, pos_z, label="Pos Z", color='blue')
plt.title("Position (X, Y, Z) vs Time")
plt.xlabel("Time (s)")
plt.ylabel("Position")
plt.legend()
plt.grid(True)

# Plot Velocity x-y-z
ax2 = plt.subplot(3, 1, 2, sharex=ax1)
plt.plot(time, vel_x, label="Vel X", color='red')
plt.plot(time, vel_y, label="Vel Y", color='green')
plt.plot(time, vel_z, label="Vel Z", color='blue')
plt.title("Velocity (X, Y, Z) vs Time")
plt.xlabel("Time (s)")
plt.ylabel("Velocity")
plt.legend()
plt.grid(True)

# Plot Wheelspeeds for each wheel.
ax3 = plt.subplot(3, 1, 3, sharex=ax1)
plt.plot(time, wheelFR_speed, label="Wheel FR", color='magenta')
plt.plot(time, wheelFL_speed, label="Wheel FL", color='cyan')
plt.plot(time, wheelRR_speed, label="Wheel RR", color='orange')
plt.plot(time, wheelRL_speed, label="Wheel RL", color='brown')
plt.title("Wheel Speeds vs Time")
plt.xlabel("Time (s)")
plt.ylabel("Wheel Speed")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
time = log_data['time']
throttle = log_data['throttle']
turboBoost = log_data['turboBoost']
superchargerBoost = log_data['superchargerBoost']

plt.figure(figsize=(12, 10))

ax1 = plt.subplot(3, 1, 1)
plt.plot(time, throttle, label="Throttle", color='blue')
plt.ylabel("Throttle")
plt.legend()
plt.grid(True)

ax2 = plt.subplot(3, 1, 2, sharex=ax1)
plt.plot(time, turboBoost, label="Turbo Boost (PSI)", color='red')
plt.ylabel("Turbo Boost (PSI)")
plt.legend()
plt.grid(True)

ax3 = plt.subplot(3, 1, 3, sharex=ax1)
plt.plot(time, superchargerBoost, label="Supercharger Boost (PSI)", color='magenta')
plt.xlabel("Time (s)")
plt.ylabel("Supercharger Boost (PSI)")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Extract data from log_data
time = log_data['time']
tvalid = np.logical_and(time < 100, time > 96)
time = time[tvalid]
rpm = log_data['RPM'][tvalid]
wheelRR_speed = np.array(log_data['wheelRR_angVel'])[tvalid]
wheelRL_speed = np.array(log_data['wheelRL_angVel'])[tvalid]

# Convert engine RPM to rad/s (rad/s = RPM * 2π/60)
engine_rad_s = rpm * (2 * np.pi / 60)

# Calculate average rear wheel speed (rad/s assumed in data)
avg_rear_wheel_speed = (wheelRR_speed + wheelRL_speed) / 2.0
ratio = engine_rad_s / avg_rear_wheel_speed
max_ratio = np.max(ratio)
# ratio = ratio / max_ratio
est_ratio = log_data['gearRatio'][tvalid]
max_est_ratio = np.max(est_ratio)
# est_ratio = est_ratio / max_est_ratio

plt.figure(figsize=(12, 6))
plt.plot(time, ratio, label="Estimated Gear Ratio", color='blue')
plt.plot(time, est_ratio, label="Actual Gear Ratio", color='red')


# plt.figure(figsize=(12, 6))
# plt.plot(time, avg_rear_wheel_speed, label="Avg Rear Wheel Speed (rad/s)", color='blue')
# plt.plot(time, engine_rad_s, label="Engine Speed (rad/s)", color='red', linestyle='--')
# plt.xlabel("Time (s)")
# plt.ylabel("Angular Speed (rad/s)")
# plt.title("Average Rear Wheel Speed vs. Engine Speed")
# plt.legend()
# plt.grid(True)
# plt.show()

In [ ]:
# Extract time and wheel angular velocity data from log_data
time = log_data['time']
# wheelRR_angVel = log_data['RPM'] * (2 * np.pi / 60)
wheelRR_angVel = log_data['wheelRR_angVel']
wheelRR_angVelB = log_data['wheelRR_angVelB']
# wheelRR = log_data['wheelRR_speed'] / 0.395

plt.figure(figsize=(12, 6))
plt.plot(time, wheelRR_angVel, label="WheelRR_angVel", color='blue')
plt.plot(time, wheelRR_angVelB, label="WheelRR_angVelB", color='red', linestyle='--')
# plt.plot(time, wheelRR, label="WheelRR_speed", color='green', linestyle='-.')
plt.xlabel("Time (s)")
plt.ylabel("Angular Velocity (rad/s)")
plt.title("Rear Right Wheel Angular Velocities")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Data extraction (using tvalid as defined in your file)
time = log_data['time']
tvalid = np.logical_and(time < 350, time > 96)
time = time[tvalid]

rpm = log_data['RPM'][tvalid]
# For rear wheel speeds, we use the angular velocity measured at the rear wheels (using angVelB keys)
wheelRR_speed = np.array(log_data['wheelRR_angVelB'])[tvalid]
wheelRL_speed = np.array(log_data['wheelRL_angVelB'])[tvalid]

# Convert engine RPM to rad/s (rad/s = RPM * 2π/60)
engine_rad_s = rpm * (2 * np.pi / 60)

# Calculate average rear wheel speed (assumed already in rad/s)
avg_rear_wheel_speed = (wheelRR_speed + wheelRL_speed) / 2.0

plt.figure(figsize=(12, 8))

# Engine Speed subplot
ax1 = plt.subplot(2, 1, 1)
plt.plot(time, engine_rad_s, label="Engine Speed (rad/s)", color='red')
plt.ylabel("Engine Speed (rad/s)")
plt.title("Engine Speed and Rear Wheel Speed vs. Time")
plt.legend()
plt.grid(True)

# Rear Wheel Speed subplot, sharing the same time (x-axis)
ax2 = plt.subplot(2, 1, 2, sharex=ax1)
plt.plot(time, avg_rear_wheel_speed, label="Avg Rear Wheel Speed (rad/s)", color='blue')
plt.xlabel("Time (s)")
plt.ylabel("Wheel Speed (rad/s)")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
time_full = log_data['time']
clutch = log_data['clutch']  # Replace with 'clutchInput' if needed

plt.figure(figsize=(12, 6))
plt.plot(time_full, clutch, label="Clutch", color='green')
plt.xlabel("Time (s)")
plt.ylabel("Clutch Value")
plt.title("Clutch vs. Time (Full Data)")
plt.legend()
plt.grid(True)
plt.show()